# Wstęp
Poniższy przykład prezentuje trening własnej sieci konwolucyjnej (CNN) do klasyfikacji obrazów. Wykorzystana zostanie biblioteka TensorFlow do stworzenia i trenowania modelu.  

Dane treningowe to zbiór cyfr MNIST, który zawiera 60,000 obrazów treningowych i 10,000 obrazów testowych przedstawiających cyfry od 0 do 9.

Do uruchomienia tego przykładu potrzebne są następujące biblioteki:
- tensorflow
- matplotlib
- scikit-learn (sklearn)
oraz środowisko Python w wersji 3.12 (wyższe wersje nie są kompatybilne z aktualnym TensorFlow).

Proponowane instalacja Miniconda lub Anaconda, które ułatwiają zarządzanie środowiskami i pakietami Pythona.

```bash
conda create -n cnn_env python=3.12
conda activate cnn_env
conda install tensorflow matplotlib scikit-learn (lub pip install tensorflow matplotlib scikit-learn)
```


# Import bibliotek

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Activation
from sklearn.metrics import confusion_matrix, classification_report

# Pobieranie danych
- Import danych
- Konwersja obrazów na float (z zakresu 0-255 na 0-1)
- Zmiana wektora wynikowego na kategorie (jeden z 10)
- Informacje o zbiorze treningowym (i testowym)
- Wydruk próbek ze zbioru treningowego

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train, X_test = X_train / 255.0, X_test / 255.0

y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

print("Rozmiar danych treningowych:", X_train.shape)
print("Rozmiar etykiet treningowych:", y_train.shape)
print("Rozmiar danych testowych:", X_test.shape)
print("Rozmiar etykiet testowych:", y_test.shape)

rows = 10
cols = 10

fig, axes = plt.subplots(rows, cols, figsize=(10, 10))
axes = axes.ravel()
for i in range(rows * cols):
    axes[i].imshow(X_train[i], cmap='gray')
    axes[i].set_title(f"{y_train[i].argmax()}")
    axes[i].axis('off')
plt.subplots_adjust(hspace=0.5)
plt.show()



## Sposób wygenerowania wartości początkowych parametrów nuronów (wag)

Przykład różnych metod inicjalizacji wag w sieciach neuronowych:
- `RandomNormal`: Inicjalizacja wag z rozkładu normalnego o średniej 0 i odchyleniu standardowym 0.05.
- `Constant`: Inicjalizacja wag stałą wartością 0.1.
- `GlorotUniform`: Inicjalizacja wag z rozkładu jednostajnego, który jest skalowany w zależności od liczby wejść i wyjść warstwy.


In [ ]:
constant_initializer = tf.keras.initializers.Constant(value=0.1)
random_initializer = tf.keras.initializers.RandomNormal(mean=0., stddev=1.)
default_initializer = tf.keras.initializers.GlorotUniform()

kernel_initializer = random_initializer

## Projekt sieci neuronowej

In [ ]:
model = Sequential()
model.add(Conv2D(filters = 32, kernel_size = 5, strides = 1, activation = 'relu', input_shape = (28,28,1)))
model.add(Conv2D(filters = 32, kernel_size = 5, strides = 1, use_bias=False))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size = 2, strides = 2))
model.add(Dropout(0.25))
model.add(Conv2D(filters = 64, kernel_size = 3, strides = 1, activation = 'relu'))
model.add(Conv2D(filters = 64, kernel_size = 3, strides = 1, use_bias=False))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size = 2, strides = 2))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(units = 256, use_bias=False))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dense(units = 128, use_bias=False))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dense(units = 84, use_bias=False))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.25))
model.add(Dense(units = 10, activation = 'softmax'))

## Kompilacja modelu i podsumowanie architektury

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## Określenie hiperparametrów i trenowanie modelu

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=10, verbose=1,callbacks=early_stopping)
print("Test danych treningowych\n", model.predict(X_train))

## Ewaluacja modelu i wyświetlenie wyników

In [ ]:
# print('Ewaluacja modelu:', model.evaluate(X_train, y_train))
fig, axs = plt.subplots(1, 2, figsize=(15, 6))

axs[0].plot(history.history['loss'])
axs[0].set_title('Wykres straty / epoki')
axs[0].set_xlabel('Epoka')
axs[0].set_ylabel('Strata')
# axs[0].set_ylim([0, 1])

axs[1].plot(history.history['accuracy'])
axs[1].set_title('Wykres dokładności / epoki')
axs[1].set_xlabel('Epoka')
axs[1].set_ylabel('Dokładność')
# axs[1].set_ylim([0, 1.2])

plt.show()

## Zapisanie modelu Keras do pliku (TensorFlow SavedModel)

Format TensorFlow SavedModel przechowuje pełną informację o modelu, a nie tylko same wagi.

In [ ]:
model_name = "./tensorflow/cyfry-mnist-test"
model.export(model_name)

## Konwersja modelu do formatu ONNX

In [ ]:
import shutil

!python -m tf2onnx.convert --saved-model {model_name} --output {model_name}.onnx
shutil.rmtree(model_name)
